In [1]:
import requests

url = "https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/40381/hscard"

headers = {
	"x-rapidapi-key": "7f1f883aeamsh791bcb5113cd18dp167a5djsnd7fe5f833574",
	"x-rapidapi-host": "cricbuzz-cricket.p.rapidapi.com"
}

response = requests.get(url, headers=headers)

data = response.json()
data

{'scorecard': [{'inningsid': 1,
   'batsman': [{'id': 1114,
     'balls': 5,
     'runs': 5,
     'fours': 1,
     'sixes': 0,
     'strkrate': '100',
     'name': 'Stirling',
     'nickname': '',
     'iscaptain': False,
     'iskeeper': False,
     'outdec': 'lbw b Nisarg Patel',
     'videotype': '',
     'videourl': '',
     'videoid': 0,
     'planid': 0,
     'imageid': 0,
     'premiumvideourl': '',
     'iscbplusfree': False,
     'ispremiumfree': False,
     'inmatchchange': '',
     'isoverseas': False,
     'playingxichange': ''},
    {'id': 6710,
     'balls': 5,
     'runs': 10,
     'fours': 2,
     'sixes': 0,
     'strkrate': '200',
     'name': 'Balbirnie',
     'nickname': '',
     'iscaptain': True,
     'iskeeper': False,
     'outdec': 'c Sushant Modani b Netravalkar',
     'videotype': '',
     'videourl': '',
     'videoid': 0,
     'planid': 0,
     'imageid': 0,
     'premiumvideourl': '',
     'iscbplusfree': False,
     'ispremiumfree': False,
     'inmatchch

In [2]:
data.keys()

dict_keys(['scorecard', 'ismatchcomplete', 'appindex', 'status', 'responselastupdated'])

In [18]:
import json
print(json.dumps(data, indent=4))

{
    "scorecard": [
        {
            "inningsid": 1,
            "batsman": [
                {
                    "id": 1114,
                    "balls": 5,
                    "runs": 5,
                    "fours": 1,
                    "sixes": 0,
                    "strkrate": "100",
                    "name": "Stirling",
                    "nickname": "",
                    "iscaptain": false,
                    "iskeeper": false,
                    "outdec": "lbw b Nisarg Patel",
                    "videotype": "",
                    "videourl": "",
                    "videoid": 0,
                    "planid": 0,
                    "imageid": 0,
                    "premiumvideourl": "",
                    "iscbplusfree": false,
                    "ispremiumfree": false,
                    "inmatchchange": "",
                    "isoverseas": false,
                    "playingxichange": ""
                },
                {
                    "id": 67

In [19]:
{
  "scoreCard": [
    {
      "score": "250/3",
      "overs": "42.1",
      "batTeamName": "India"
    }
  ]
}


{'scoreCard': [{'score': '250/3', 'overs': '42.1', 'batTeamName': 'India'}]}

In [24]:
card = data["scorecard"][0]

team = card.get("batteamname")
score = card.get("score")
overs = card.get("overs")
print(f"Team_Name :- {team}\nTeam_Score :- {score}\nOver :- {overs}")

Team_Name :- Ireland
Team_Score :- 150
Over :- 18.5


In [25]:
card = data["scorecard"][1]

team = card.get("batteamname")
score = card.get("score")
overs = card.get("overs")
print(f"Team_Name :- {team}\nTeam_Score :- {score}\nOver :- {overs}")

Team_Name :- United States
Team_Score :- 141
Over :- 20.0


In [2]:
# app.py

"""
cricket-live-score - Flask app to save match IDs and fetch live score via Cricbuzz RapidAPI.

Features:
- Home page to add a match ID and list saved match IDs from SQLite.
- Click a saved match to open match page which calls Cricbuzz API:
  https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_id}/hscard
- Database (SQLite) auto-created as score.db
- Basic error handling and clean UI rendering

How to run:
1. pip install -r requirements.txt
2. python app.py
3. Open http://127.0.0.1:5000 in browser
"""

from flask import Flask, render_template, request, redirect, url_for, flash
import sqlite3
import requests
import json
import os

APP_DIR = os.path.dirname(os.path.abspath(__file__))
DB_PATH = os.path.join(APP_DIR, "score.db")

RAPIDAPI_KEY = "7f1f883aeamsh791bcb5113cd18p167a5djsnd7fe5f833574"
RAPIDAPI_HOST = "cricbuzz-cricket.p.rapidapi.com"

app = Flask(__name__)
app.secret_key = "supersecretkey_change_this"  

def init_db():
    """Create database and matches table if not exists."""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("""
        CREATE TABLE IF NOT EXISTS matches (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            match_id TEXT UNIQUE NOT NULL,
            added_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    conn.close()

init_db()

def add_match_to_db(match_id):
    """Insert match_id into DB; ignore duplicates."""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    try:
        c.execute("INSERT INTO matches (match_id) VALUES (?)", (match_id.strip(),))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

def get_all_matches():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT id, match_id, added_at FROM matches ORDER BY added_at DESC")
    rows = c.fetchall()
    conn.close()
    return [{"id": r[0], "match_id": r[1], "added_at": r[2]} for r in rows]

@app.route("/", methods=["GET", "POST"])
def home():
    """
    Home page:
    - GET: show form to input a match ID and list saved match IDs
    - POST: save new match ID to DB
    """
    if request.method == "POST":
        match_id = request.form.get("match_id", "").strip()
        if not match_id:
            flash("Please enter a valid Match ID.", "danger")
            return redirect(url_for("home"))

        added = add_match_to_db(match_id)
        if added:
            flash(f"Match ID {match_id} added successfully.", "success")
        else:
            flash(f"Match ID {match_id} is already saved.", "info")

        return redirect(url_for("home"))

    saved = get_all_matches()
    return render_template("home.html", matches=saved)

@app.route("/match/<match_id>")
def match_page(match_id):
    """
    Match page:
    - Calls Cricbuzz RapidAPI to fetch match data for given match_id
    - Shows raw JSON pretty-printed and a small summary if available
    """
    url = f"https://cricbuzz-cricket.p.rapidapi.com/mcenter/v1/{match_id}/hscard"
    headers = {
        "x-rapidapi-key": "7f1f883aeamsh791bcb5113cd18dp167a5djsnd7fe5f833574",
        "x-rapidapi-host": RAPIDAPI_HOST,
        "Accept": "application/json"
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        status_code = response.status_code
        if status_code == 200:
            try:
                data = response.json()
            except json.JSONDecodeError:
                data = {"raw_text": response.text}
            pretty_json = json.dumps(data, indent=2, ensure_ascii=False)
            summary = {}
            if isinstance(data, dict):
                if "match" in data and isinstance(data["match"], dict):
                    m = data["match"]
                    summary["match_name"] = m.get("matchDesc") or m.get("matchTitle") or m.get("name")
                if "team" in data and isinstance(data["team"], list):
                    teams = [t.get("teamNameShort") or t.get("team_name") or t.get("name") for t in data["team"]]
                    summary["teams"] = [t for t in teams if t]
                if "ms" in data:
                    summary["match_state"] = data.get("ms")
                if "score" in data:
                    summary["score"] = data.get("score")
            return render_template("match.html", match_id=match_id, data=pretty_json, summary=summary, status=status_code)
        else:
            text = response.text
            error_msg = f"API returned status {status_code}: {text}"
            return render_template("match.html", match_id=match_id, data=json.dumps({"error": error_msg}, indent=2), summary={}, status=status_code)

    except requests.RequestException as e:
        err = {"error": f"Request failed: {str(e)}"}
        return render_template("match.html", match_id=match_id, data=json.dumps(err, indent=2), summary={}, status="request-failed")

@app.route("/delete/<int:row_id>", methods=["POST"])
def delete_match(row_id):
    """Delete saved match by DB row id."""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("DELETE FROM matches WHERE id = ?", (row_id,))
    conn.commit()
    conn.close()
    flash("Saved match removed.", "success")
    return redirect(url_for("home"))

if __name__ == "__main__":
    app.run(debug=True)


In [3]:
{% extends "base.html" %}
{% block content %}
  <h2>Match Details — ID: {{ match_id }}</h2>

  <section class="card">
    <div class="summary">
      <h3>Summary</h3>
      {% if summary and summary.items()|length > 0 %}
        <ul>
          {% for k, v in summary.items() %}
            <li><strong>{{ k|replace('_', ' ')|title }}:</strong>
              {% if v is iterable and v is not string %}
                {{ v|join(', ') }}
              {% else %}
                {{ v }}
              {% endif %}
            </li>
          {% endfor %}
        </ul>
      {% else %}
        <p>No clean summary available — raw API response shown below.</p>
      {% endif %}
    </div>

    <div class="api-status">
      <strong>API status:</strong> {{ status }}
    </div>

    <div class="raw-data">
      <h3>API Raw JSON</h3>
      <pre>{{ data }}</pre>
    </div>

    <div class="actions">
      <a class="btn" href="{{ url_for('home') }}">Back</a>
      <a class="btn" href="{{ url_for('match_page', match_id=match_id) }}">Refresh</a>
    </div>
  </section>
{% endblock %}
